# Stage F: Block Power Method for Alcaraz DQPT Characterization

## Motivation: why Z2 sector projection failed

Notebook 15 (Stage C) extracted the Z2 representation $R$ on the temporal physical indices
of the tMPO bulk tensor $W_c$. The representation is a $7 \times 7$ real involution with
eigenvalues $\{+1, +1, +1, +1, -1, -1, -1\}$ — four even and three odd channels.

**Key finding:** the tMPO boundary tensors $W_l$ and $W_r$ (built from the initial state
$|\psi_0\rangle = |X^+\rangle^{\otimes N}$) live **entirely in the even sector** of $R$.
The FSM start channel (1) and end channel (7) both have $R$-eigenvalue $+1$, so the boundary
vector $L = [1, 0, \ldots, 0, 1]$ only populates even-parity bond channels.

**Consequence:** the full tMPO (with boundaries) has zero support in the odd sector.
Projecting the tMPS into the odd sector and applying this tMPO gives exactly zero.
This is physically correct — the Loschmidt echo $L(t) = \langle X^+ | e^{-iHt} | X^+ \rangle$
only probes the even parity sector because $|X^+\rangle$ is a $+1$ eigenstate of
$P = \prod_i \sigma^x_i$.

## Intra-sector degeneracy at the DQPT

The convergence failure of the single-vector power method at $T \geq 6$ (Stage A: $\Delta S$
plateau at $\sim 3 \times 10^{-2}$) is **not** caused by even/odd sector degeneracy. It must
be caused by near-degeneracy **within the even sector** — the standard DQPT mechanism:

At a **dynamical quantum phase transition** (DQPT), the dominant transfer-matrix eigenvalue
$\lambda_0(T)$ undergoes a phase crossing: $\arg(\lambda_0)$ jumps by $\sim \pi$, and a
subleading eigenvalue $\lambda_1(T)$ (also in the even sector) temporarily becomes comparable
in magnitude: $|\lambda_1| / |\lambda_0| \to 1$. This closes the spectral gap and makes the
single-vector power method unable to converge — it oscillates between the two competing
eigenspaces.

## The block power method: oblique Rayleigh-Ritz

The solution is a **multi-vector (block/subspace) power method** that extracts the leading
$k$ eigenvalues simultaneously. The algorithm (`block_transfer_eigs` in `dqpt_diagnostics.jl`)
uses an **oblique Petrov-Galerkin** scheme:

1. Maintain $k$ right tMPS $\{|R_j\rangle\}$ and $k$ left tMPS $\{\langle L_i|\}$
2. Each iteration: apply the tMPO to all $k$ vectors (right: $A|R_j\rangle$, left: $A^T\langle L_i|$)
3. Build the $k \times k$ oblique pencil:
   - Overlap matrix: $S_{ij} = \langle L_i | R_j \rangle$ (non-conjugating)
   - Action matrix: $M_{ij} = \langle L_i | A | R_j \rangle$
4. Solve the generalized eigenvalue problem $S^{-1} M \mathbf{v} = \lambda \mathbf{v}$
   via `pinv(S) * M` (pseudo-inverse handles near-singular $S$ at the gap closing)
5. Rotate the basis by the Ritz vectors to de-mix the $k$ eigenspaces

This produces $k$ Ritz values $\{\theta_0, \theta_1, \ldots, \theta_{k-1}\}$ (sorted by
$|\theta|$), which are the leading transfer-matrix eigenvalues. Near the DQPT, $|\theta_0|
\approx |\theta_1|$ but $\arg(\theta_0) \neq \arg(\theta_1)$ — the block method captures both
without confusion.

## What we expect

- $|\lambda_j| \leq 1$ for all $j$ and all $T$ (unitarity bound)
- Gap ratio $|\lambda_1|/|\lambda_0| \to 1$ near the DQPT time $T^*$
- Phase crossing: $\arg(\lambda_0)$ and $\arg(\lambda_1)$ exchange near $T^*$
- The Loschmidt rate $\ell(T) = -\log|\lambda_0(T)|$ should be a smooth function with a kink at $T^*$

In [1]:
using ITensors, ITensorMPS, ITransverse, ProgressMeter
using JLD2, Plots, LinearAlgebra, Printf
ProgressMeter.ijulia_behavior(:clear)

include("main.jl")
include("dqpt_diagnostics.jl")

# Parameters (consistent with Stages A-C)
p       = 0.1
lambda  = 1.0
dt      = 0.1
maxdim  = 256
cutoff  = 1e-12

println("Alcaraz model: p=$p, λ=$lambda, dt=$dt, maxdim=$maxdim")

Alcaraz model: p=0.1, λ=1.0, dt=0.1, maxdim=256


## Block PM algorithm details

The implementation is in `dqpt_diagnostics.jl:block_transfer_eigs`. Key design choices:

**Non-Hermitian two-sided iteration.** The transfer matrix $T$ of the tMPO is non-Hermitian
(it arises from a non-unitary contraction of the 2D tensor network). We maintain separate
left and right bases: the right vectors $|R_j\rangle$ are updated by applying $T$, and the
left vectors $\langle L_i|$ by applying $T^T$ (pure transpose, **no** complex conjugation —
the bilinear-form adjoint, not the Hermitian adjoint).

**Generalized eigenvalue problem via pseudo-inverse.** The overlap matrix
$S_{ij} = \langle L_i | R_j \rangle$ (computed with `overlap_noconj`, never `inner`) can
become near-singular when two eigenvalues are nearly degenerate. We solve $W = S^{-1}M$ using
`pinv(S; rtol=1e-12)` instead of `inv(S)` or `eigen(M, S)` — both of which diverge or throw
`SingularException` at the gap closing.

**Basis refresh.** When $\text{cond}(S) > 10^{10}$, the $k$-th basis direction is replaced
with a fresh random tMPS, Gram-Schmidt orthogonalized against the first $k-1$ directions
using `overlap_noconj` projections. This prevents numerical collapse when two Ritz vectors
converge to the same eigenspace.

**MPS linear combinations.** The Ritz-vector rotation $R_j^{\text{new}} = \sum_i V_{ij}^R \cdot
A|R_i\rangle$ uses `lincomb_mps` (exact directsum + SVD truncation), not the default density-matrix
`+` which NaN-crashes on near-degenerate sums.

**Parameters for this run:**
- $k = 4$ eigenvalues (captures the leading pair + 2 subleading for diagnostics)
- `itermax = 800` (more room for the gap-closing regime at $T \geq 5$)
- `eps_conv = 1e-6` (relaxed; sufficient for rate/phase extraction near the DQPT)
- `n_track = 2` (converge on the leading 2 eigenvalues only)

Near the DQPT, the block PM may hit `itermax` without fully converging. The Ritz values at
`maxiter` are still informative — the oblique pencil gives much better eigenvalue estimates
than the single-vector Rayleigh quotient even when not fully converged.

In [ ]:
# Block PM sweep — crash-safe, caches per T
# NOTE: T≤5.0 already cached with eps_conv=1e-7.
# For T≥5.5 we relax convergence: the gap closes and exact convergence
# would take thousands of iterations. The Ritz values at maxiter are still
# informative — the block PM's pencil gives much better estimates than the
# single-vector Rayleigh quotient even when not fully converged.
T_grid    = collect(0.5:0.5:8.0)
k_block   = 4
itermax   = 800
eps_conv  = 1e-6      # relaxed from 1e-7; sufficient for rate/phase extraction
cachefile = "block_pm_alcaraz_p$(p).jld2"

done = crashsafe_sweep(T_grid; cachefile=cachefile) do T
    @info "Block PM: T=$T"
    mpo, scaffold = build_alcaraz_tmpo(T; p=p, lambda=lambda, dt=dt)
    theta, Lvecs, Rvecs, info = block_transfer_eigs(mpo, scaffold;
        k=k_block, maxdim=maxdim, cutoff=cutoff,
        itermax=itermax, eps_conv=eps_conv, n_track=2)

    # Store the eigenvalues and diagnostics (not the full MPS — too large for JLD2)
    (theta       = collect(theta),
     niters      = info[:niters],
     reason      = string(info[:reason]),
     condS       = info[:condS],
     dtheta_hist = info[:dtheta],
     condS_hist  = info[:condS_hist])
end

# Summary table
Ts_sorted = sort(collect(keys(done)))
println("\n" * "="^80)
@printf("%-6s  %-10s %-10s %-10s %-10s %-8s %-6s  %s\n",
        "T", "|θ₀|", "|θ₁|", "gap_ratio", "arg(θ₀)", "niters", "cond", "reason")
println("-"^80)
for T in Ts_sorted
    r = done[T]
    haskey(r, :error) && (println(@sprintf("%-6.1f  ERROR: %s", T, r.error)); continue)
    th = r.theta
    gr = abs(th[2]) / max(abs(th[1]), 1e-30)
    @printf("%-6.1f  %-10.5f %-10.5f %-10.5f %-10.4f %-8d %-6.1e  %s\n",
            T, abs(th[1]), abs(th[2]), gr, angle(th[1]), r.niters, r.condS, r.reason)
end

In [ ]:
# 3-panel spectral gap plot
Ts_plot = Float64[]
abs_th  = [Float64[] for _ in 1:k_block]
gap_rat = Float64[]
arg_th0 = Float64[]
arg_th1 = Float64[]
converged_mask = Bool[]

for T in sort(collect(keys(done)))
    r = done[T]
    haskey(r, :error) && continue
    th = r.theta
    any(isnan, abs.(th)) && continue
    push!(Ts_plot, T)
    for j in 1:min(k_block, length(th))
        push!(abs_th[j], abs(th[j]))
    end
    push!(gap_rat, abs(th[2]) / max(abs(th[1]), 1e-30))
    push!(arg_th0, angle(th[1]))
    push!(arg_th1, angle(th[2]))
    push!(converged_mask, r.reason == "converged")
end

p1 = plot(title="Leading transfer-matrix eigenvalues", xlabel="T", ylabel="|λ|")
colors = [:black, :red, :orange, :gray]
labels = ["|λ₀|", "|λ₁|", "|λ₂|", "|λ₃|"]
for j in 1:k_block
    plot!(p1, Ts_plot, abs_th[j]; label=labels[j], color=colors[j],
          marker=:circle, ms=4, lw=1.5)
end
hline!(p1, [1.0]; color=:gray, ls=:dash, label="", alpha=0.5)

p2 = plot(Ts_plot, gap_rat; title="Gap ratio (→1 ⇒ DQPT)", xlabel="T",
          ylabel="|λ₁|/|λ₀|", color=:purple, marker=:circle, ms=4, lw=1.5, label="")
hline!(p2, [1.0]; color=:gray, ls=:dash, label="", alpha=0.5)

p3 = plot(title="Eigenvalue phases", xlabel="T", ylabel="arg(λ)")
plot!(p3, Ts_plot, arg_th0; label="arg(λ₀)", color=:black, marker=:circle, ms=4, lw=1.5)
plot!(p3, Ts_plot, arg_th1; label="arg(λ₁)", color=:red,   marker=:circle, ms=4, lw=1.5)
hline!(p3, [π, -π]; color=:gray, ls=:dash, label="±π", alpha=0.3)

plt = plot(p1, p2, p3; layout=(1,3), size=(1500,480), margin=5Plots.mm,
           plot_title="Alcaraz block PM spectral gap (p=$p, λ=$lambda, dt=$dt)")
mkpath("imgs")
savefig(plt, "imgs/block_pm_alcaraz_p$(p).png")
plt

## TDVP comparison: transverse vs Schrödinger Loschmidt rate

The transverse contraction and TDVP compute the Loschmidt echo in fundamentally different ways:

**Transverse (thermodynamic limit).** The transfer matrix $T$ of the tMPO encodes one spatial
column of the 2D network. The Loschmidt amplitude for a chain of $N$ sites is:
$$L(T) = \langle \text{bl} | T^N | \text{br} \rangle \approx \lambda_0^N \, c_L \, c_R$$
where $\lambda_0$ is the dominant eigenvalue and $c_{L,R}$ are boundary overlaps. The intensive
Loschmidt rate in the thermodynamic limit is:
$$\ell_\infty(T) = -\log|\lambda_0(T)|$$
This is a **per-column** (= per spatial site) quantity, independent of $N$.

**TDVP (finite $N$).** Direct Schrödinger evolution of $|\psi(t)\rangle = e^{-iHt}|\psi_0\rangle$
via the TDVP algorithm on an MPS of $N$ sites. The Loschmidt amplitude is
$G(T) = \langle \psi_0 | \psi(T) \rangle$ (complex, with `inner`). The intensive rate is:
$$\ell_N(T) = -\frac{\log|G(T)|}{N}$$

**Expected relation:** $\ell_N(T) \to \ell_\infty(T) + O(1/N)$ as $N \to \infty$. The $O(1/N)$
correction comes from the boundary overlaps $c_{L,R}$ and from the open-chain boundary conditions.

**Phase comparison:** $\arg(\lambda_0(T))$ (transverse, per column) vs $\arg(G(T))$ (TDVP, full
chain). For large $N$: $\arg(G) \approx N \cdot \arg(\lambda_0) + \arg(c_L c_R) \pmod{2\pi}$.
We compare $\arg(\lambda_0)$ against $\arg(G)/N$ (modulo $2\pi/N$ aliasing).

**Existing TDVP data:**
- $N=40$, $\delta t = 0.05$: $T \in \{0.5, 1.0, 1.5, \ldots, 7.0\}$ — cached in `tdvp_loschmidt_p0.1_N40.jld2`
- $N=80$, $\delta t = 0.05$: $T \in \{0.5, 1.0, \ldots, 4.0\}$ — cached in `tdvp_loschmidt_p0.1_N80.jld2`

We will extend the $N=80$ data to $T=7$ to match the transverse sweep range.

In [ ]:
# Load TDVP data
tdvp_40 = load("tdvp_loschmidt_p$(p)_N40.jld2", "done")
tdvp_80 = load("tdvp_loschmidt_p$(p)_N80.jld2", "done")

# Extend N=80 to T=7 (slow — ~hours; comment out if already cached)
# Uncomment and run if needed:
# tdvp_80 = tdvp_loschmidt_amplitude(80, collect(0.5:0.5:7.0);
#     p=p, lambda=lambda, dt=0.05, cutoff=1e-12, maxdim=256)

# Extract arrays for plotting
function extract_tdvp(d)
    Ts = sort(collect(keys(d)))
    rates = [d[T].rate for T in Ts]
    phases = [angle(d[T].G) for T in Ts]
    absG   = [d[T].absG for T in Ts]
    return Ts, rates, phases, absG
end

Ts40, rate40, phase40, absG40 = extract_tdvp(tdvp_40)
Ts80, rate80, phase80, absG80 = extract_tdvp(tdvp_80)

# Transverse rate from block PM
rate_trans = [-log(max(abs(done[T].theta[1]), 1e-30)) for T in Ts_plot if !haskey(done[T], :error)]
Ts_trans   = [T for T in Ts_plot if !haskey(done[T], :error)]

# Comparison plots
p1 = plot(title="Loschmidt rate ℓ(T)", xlabel="T", ylabel="ℓ(T) = -log|λ₀| or -log|G|/N")
plot!(p1, Ts_trans, rate_trans; label="Transverse (block PM)", color=:blue,
      marker=:circle, ms=5, lw=2)
plot!(p1, Ts40, rate40; label="TDVP N=40", color=:green, marker=:diamond, ms=4, lw=1.5)
plot!(p1, Ts80, rate80; label="TDVP N=80", color=:darkgreen, marker=:square, ms=4, lw=1.5, ls=:dash)

p2 = plot(title="Phase per column", xlabel="T", ylabel="arg(λ₀) or arg(G)/N")
plot!(p2, Ts_trans, arg_th0; label="Transverse arg(λ₀)", color=:blue,
      marker=:circle, ms=5, lw=2)
plot!(p2, Ts40, phase40 ./ 40; label="TDVP arg(G)/40", color=:green,
      marker=:diamond, ms=4, lw=1.5, alpha=0.7)

p3 = plot(title="|G(T)| comparison", xlabel="T", ylabel="|G|")
plot!(p3, Ts40, absG40; label="TDVP |G| (N=40)", color=:green, marker=:diamond, ms=4, lw=1.5)
# Transverse prediction: |G| ≈ |λ₀|^N (ignoring boundary overlaps)
absG_trans_pred = [abs(done[T].theta[1])^40 for T in Ts_trans]
plot!(p3, Ts_trans, absG_trans_pred; label="Trans |λ₀|^40", color=:blue,
      marker=:circle, ms=4, lw=1.5, ls=:dash)

plt = plot(p1, p2, p3; layout=(1,3), size=(1500,480), margin=5Plots.mm,
           plot_title="Block PM vs TDVP — Alcaraz p=$p, λ=$lambda")
savefig(plt, "imgs/block_pm_vs_tdvp_p$(p).png")
plt

## Convergence diagnostics

The block PM tracks two quantities per iteration:

- **$\Delta\theta$**: $\max_{j \leq n_\text{track}} |\theta_j^{(it)} - \theta_j^{(it-1)}|$ — how
  much the leading Ritz values changed. Convergence is declared when $\Delta\theta < \epsilon_\text{conv}$.
  A decaying trajectory means the method is finding its eigenspaces; a plateau means it's stuck
  (the spectral gap is too small for the subspace to separate the eigenspaces).

- **$\text{cond}(S)$**: condition number of the overlap matrix $S_{ij} = \langle L_i | R_j \rangle$.
  Spikes when two Ritz vectors become nearly parallel (linearly dependent in the bi-orthogonal sense).
  A $\text{cond}(S) > 10^{10}$ triggers the basis refresh mechanism.

We plot $\Delta\theta$ vs iteration for selected $T$ values to assess whether the method converged
cleanly, hit `itermax`, or was saved by the refresh mechanism.

In [ ]:
# Convergence diagnostics: Δθ trajectory for selected T values
T_diag = [1.0, 4.0, 6.0, 7.0]
T_diag_available = [T for T in T_diag if haskey(done, T) && !haskey(done[T], :error)]

p1 = plot(title="Δθ convergence trajectory", xlabel="Iteration", ylabel="Δθ",
          yscale=:log10, legend=:topright)
for T in T_diag_available
    r = done[T]
    dth = r.dtheta_hist
    isempty(dth) && continue
    plot!(p1, 1:length(dth), dth; label="T=$T ($(r.reason))", lw=1.5)
end
hline!(p1, [eps_conv]; color=:red, ls=:dash, label="εconv", alpha=0.5)

p2 = plot(title="cond(S) trajectory", xlabel="Iteration", ylabel="cond(S)",
          yscale=:log10, legend=:topright)
for T in T_diag_available
    r = done[T]
    ch = r.condS_hist
    isempty(ch) && continue
    plot!(p2, 1:length(ch), ch; label="T=$T", lw=1.5)
end
hline!(p2, [1e10]; color=:red, ls=:dash, label="refresh threshold", alpha=0.5)

# Summary: final condS and niters vs T
p3 = plot(title="Final cond(S) and niters vs T", xlabel="T")
Ts_all = sort(collect(keys(done)))
Ts_ok  = [T for T in Ts_all if !haskey(done[T], :error)]
condS_final = [done[T].condS for T in Ts_ok]
niters_all  = [done[T].niters for T in Ts_ok]
plot!(p3, Ts_ok, condS_final; label="cond(S)", yscale=:log10, color=:purple,
      marker=:circle, ms=4, lw=1.5, ylabel="cond(S)")
p3b = twinx(p3)
plot!(p3b, Ts_ok, niters_all; label="niters", color=:orange, marker=:square,
      ms=4, lw=1.5, ylabel="niters", legend=:topleft)

plt = plot(p1, p2, p3; layout=(1,3), size=(1500,480), margin=5Plots.mm,
           plot_title="Block PM convergence diagnostics — Alcaraz p=$p")
savefig(plt, "imgs/block_pm_convergence_p$(p).png")
plt

## Temporal entropy extraction from block PM eigenvectors

The block PM returns bi-orthonormal left/right eigenvectors $\langle L_j |$ and $|R_j \rangle$
for each Ritz value $\theta_j$. The dominant pair $(L_0, R_0)$ defines the reduced transition
matrix (RTM) at each temporal cut:
$$\mathcal{T}_t = \frac{\text{Tr}_{\text{complement}} |R_0\rangle \langle L_0 |}{\langle L_0 | R_0 \rangle}$$

The generalized Rényi-2 temporal entropy at cut $t$ is:
$$S_2^{\text{gen}}(t) = -\log \text{Tr}(\mathcal{T}_t^2)$$

This is computed by `ITransverse.gen_renyi2(L, R)`, which returns a complex vector (one entry
per internal bond of the tMPS, i.e. $N_t - 1$ values).

**Key advantage of block PM for entropy:** At $T$ values where the single-vector PM was stuck
($T \geq 6$), the block PM's $L_0, R_0$ are properly de-mixed from $L_1, R_1$ (the subleading
eigenspace), so the entropy profile is trustworthy even near the DQPT.

**CFT prediction (Ising critical point, $c = 1/2$):** At long times after a quench to the
critical point, the temporal entropy grows logarithmically:
$$\text{Re}(S_2^{\text{gen}}) \sim \frac{c}{3} \log(t) + \text{const}$$
with $c/3 = 1/6 \approx 0.167$ for the Ising universality class. We compare against this
slope for the converged $T$ values.

**Note:** entropy extraction requires re-running the block PM to obtain the actual MPS vectors
(the crash-safe cache only stores eigenvalues and diagnostics, not the full tMPS). We run the
block PM once more for selected converged $T$ values and extract $S_2^{\text{gen}}$ directly.

In [ ]:
# Entropy extraction from block PM eigenvectors
# Re-run block PM for selected T values to get the actual MPS vectors

T_entropy = [2.0, 4.0, 6.0, 8.0]  # select T values that converged
T_entropy_ok = [T for T in T_entropy if haskey(done, T) && !haskey(done[T], :error)]

entropy_results = Dict{Float64, Any}()

for T in T_entropy_ok
    @info "Entropy extraction: T=$T"
    mpo, scaffold = build_alcaraz_tmpo(T; p=p, lambda=lambda, dt=dt)
    theta, Lvecs, Rvecs, info = block_transfer_eigs(mpo, scaffold;
        k=k_block, maxdim=maxdim, cutoff=cutoff,
        itermax=itermax, eps_conv=eps_conv, n_track=2)

    # Extract entropy from the dominant eigenvectors L[1], R[1]
    psi_L = Lvecs[1]
    psi_R = Rvecs[1]

    s2 = ITransverse.gen_renyi2(psi_L, psi_R)
    entropy_results[T] = (s2_re=real.(s2), s2_im=imag.(s2),
                          theta0=theta[1], reason=info[:reason])
    @info "  |θ₀|=$(round(abs(theta[1]),digits=5)), reason=$(info[:reason]), entropy bonds=$(length(s2))"
    GC.gc()
end

# Plot entropy profiles
p1 = plot(title="Re(S₂ᵍᵉⁿ) — block PM eigenvectors", xlabel="Temporal cut t/T", ylabel="Re(S₂)")
p2 = plot(title="Im(S₂ᵍᵉⁿ) — block PM eigenvectors", xlabel="Temporal cut t/T", ylabel="Im(S₂)")

colors_ent = cgrad(:viridis, length(T_entropy_ok), categorical=true)
for (i, T) in enumerate(sort(collect(keys(entropy_results))))
    r = entropy_results[T]
    n_bonds = length(r.s2_re)
    x = range(0.0, 1.0, length=n_bonds)
    lab = "T=$(round(T,digits=1)) ($(r.reason))"
    plot!(p1, x, r.s2_re; label=lab, color=colors_ent[i], lw=2)
    plot!(p2, x, r.s2_im; label=lab, color=colors_ent[i], lw=2)
end

# CFT reference: S₂ ~ (c/3) log(t) + const, c=1/2 for Ising
# On the normalized axis t/T ∈ [0,1], log(t) = log(T·x) = log(T) + log(x)
# We plot c/3 * log(x) shifted to match the data at x=0.5
if !isempty(entropy_results)
    x_cft = range(0.05, 0.95, length=100)
    c_ising = 0.5
    s2_cft = (c_ising / 3) .* log.(x_cft .* (1.0 .- x_cft))  # chord-length formula
    s2_cft_shifted = s2_cft .- minimum(s2_cft) .+ 0.0  # shift to start near 0
    plot!(p1, x_cft, s2_cft_shifted; label="CFT c=1/2 (shifted)", color=:black,
          ls=:dash, lw=2, alpha=0.5)
end

plt = plot(p1, p2; layout=(1,2), size=(1200,500), margin=5Plots.mm,
           plot_title="Temporal entropies — Alcaraz p=$p, λ=$lambda, dt=$dt (block PM)")
savefig(plt, "imgs/block_pm_entropies_p$(p).png")
plt

## Summary and conclusions

### Key results from this notebook

1. **Block PM convergence:** Did the block PM resolve the $T \geq 6$ convergence failure from the
   single-vector power method? Check the summary table and $\Delta\theta$ trajectories above.
   If $|\lambda_0| \leq 1$ at all $T$, the block PM succeeded where the single-vector method failed.

2. **DQPT characterization:** The spectral gap plot shows where $|\lambda_1|/|\lambda_0| \to 1$
   and the phase crossing $\arg(\lambda_0) \leftrightarrow \arg(\lambda_1)$ occurs. This identifies
   the DQPT time $T^*$ for the Alcaraz model at $p = 0.1$.

3. **Rate comparison with TDVP:** The Loschmidt rate $\ell(T)$ from the transverse block PM vs TDVP.
   An absolute offset $\ell_\infty - \ell_N \sim O(1/N)$ is expected from boundary corrections.
   The **shape** (curvature, kinks) should match between the two methods.

4. **Temporal entropies:** The $\text{Re}(S_2^{\text{gen}})$ profiles from the block PM
   eigenvectors, compared against the Ising CFT prediction $S_2 \sim (c/3) \log(t)$ with $c = 1/2$.

### Next steps

- **Vary $p$:** Repeat the block PM sweep for $p = 0, 0.2, 0.5, 1.0$ to track how the DQPT
  time $T^*$ and entropy growth depend on the NNN coupling.
- **$p = 0$ Ising control (Stage E):** At $p = 0$ the model is the integrable transverse-field
  Ising model. The transverse result should match ITransverse's built-in Ising test exactly.
- **dt convergence:** Compare $dt = 0.1$ vs $dt = 0.05$ to assess Trotter error.
- **Entropy scaling:** Extract the effective central charge $c_\text{eff}$ from the logarithmic
  slope of $\text{Re}(S_2^{\text{gen}})$ as a function of $p$. The central question: does
  $c_\text{eff} = 1/2$ (Ising) survive NNN frustration in the temporal direction?